# TFT — **Paper-Replikat: ein Modell je Station**

Diese Variante gleicht das **Versuchsprotokoll** so weit wie moeglich an das Referenz-Paper an:
**Tsokov, Lazarova & Aleksieva-Petrova (2022),** *Sustainability* **14(9):5104.**

### Was hier GLEICH gemacht wird wie im Paper

1. **Ein Modell je Station** (getrennt trainiert), fuer die drei Paper-Stationen **Dongsi, Wanliu, Changping**.
2. **Exakte Zeit-Aufteilung** (Paper, Abschnitt 3.2.3):
   - Training: 2013-03-01 bis 2015-02-28
   - Validierung: 2015-03-01 bis 2016-02-29
   - **Test: 2016-03-01 bis 2017-02-28**
   - Das finale Modell wird — wie im Paper — auf **Training + Validierung** trainiert und auf dem Test-Jahr bewertet.
     (Ein kleiner 30-Tage-Rest am Ende dient nur dem EarlyStopping; das Paper trainiert feste 100 Epochen.)
3. **Gleiche Eingabevariablen** (die GA-Auswahl des Papers, Abschnitt 3.3):
   `PM2.5, PM10, SO2, PRES, DEWP, RAIN, wd, WSPM` — **ohne** NO2, CO, O3, TEMP.
4. **Gleiche Imputation:** lineare Interpolation. Das Paper nutzt eine Hybrid-Strategie, stellt aber
   selbst fest, dass sie sich **nicht signifikant** von linearer Interpolation unterscheidet.
5. **1-Stunde-Vorhersage** (wie im Paper).

### Was UNVERMEIDBAR anders bleibt (bitte in der Praesentation nennen)

- **Modellarchitektur:** wir nutzen einen **TFT**, das Paper ein GA-optimiertes **2D-CNN + LSTM**.
- **Raeumliche Nachbar-Info:** das Paper codiert Nachbarstationen als **Bilder** (raeumliche Faltung);
  unser Per-Station-TFT nutzt **nur die eigenen Kanaele der Zielstation** (rein zeitliches Modell).
  Das ist der groesste verbleibende Unterschied.
- Kein genetischer Hyperparameter-Suchlauf, kein 10-fach-Ensemble.

Die Zahlen sind damit **so fair vergleichbar wie ohne Nachbau ihrer CNN-Architektur moeglich**.

## 1. Bibliotheken

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

pl.seed_everything(42)
torch.set_float32_matmul_precision("medium")
print("torch:", torch.__version__, "| GPU:", torch.cuda.is_available())

## 2. Daten laden (UCI-Download, sonst synthetischer Ersatz)

In [ ]:
import io, zipfile, urllib.request, os

DATA_DIR = "beijing_data"
UCI_URL = "https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip"
STATIONS = ["Aotizhongxin","Changping","Dingling","Dongsi","Guanyuan","Gucheng",
            "Huairou","Nongzhanguan","Shunyi","Tiantan","Wanliu","Wanshouxigong"]

def try_download():
    try:
        print("Lade UCI-Datensatz ...")
        rawb = urllib.request.urlopen(UCI_URL, timeout=30).read()
        z = zipfile.ZipFile(io.BytesIO(rawb)); os.makedirs(DATA_DIR, exist_ok=True)
        for name in z.namelist():
            if name.endswith(".zip"):
                zipfile.ZipFile(io.BytesIO(z.read(name))).extractall(DATA_DIR)
            elif name.endswith(".csv"):
                z.extract(name, DATA_DIR)
        print("Download OK."); return True
    except Exception as e:
        print("Download fehlgeschlagen:", e); return False

def load_csvs():
    import glob
    files = glob.glob(os.path.join(DATA_DIR, "**", "PRSA_Data_*.csv"), recursive=True)
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if len(files) >= 12 else None

def make_synthetic():
    print(">>> SYNTHETISCHE Demo-Daten (nur zum Testen!) <<<")
    rng = np.random.default_rng(0); frames = []
    hours = pd.date_range("2013-03-01", "2017-02-28 23:00", freq="h")
    for st in STATIONS:
        n = len(hours); daily = 30*np.sin(2*np.pi*hours.hour/24)
        frames.append(pd.DataFrame({
            "year":hours.year,"month":hours.month,"day":hours.day,"hour":hours.hour,
            "PM2.5":(80+daily+rng.normal(0,15,n)).clip(1),"PM10":(100+daily+rng.normal(0,20,n)).clip(1),
            "SO2":rng.normal(15,5,n).clip(0),"NO2":rng.normal(50,15,n).clip(0),"CO":rng.normal(1000,300,n).clip(0),
            "O3":rng.normal(60,20,n).clip(0),"TEMP":13+10*np.sin(2*np.pi*hours.dayofyear/365)+rng.normal(0,2,n),
            "PRES":rng.normal(1012,8,n),"DEWP":rng.normal(2,5,n),"RAIN":0.0,
            "wd":rng.choice(["N","NE","E","SE","S","SW","W","NW"],n),"WSPM":rng.normal(2,1,n).clip(0),"station":st}))
    return pd.concat(frames, ignore_index=True)

raw = load_csvs() if try_download() else None
if raw is None:
    raw = make_synthetic()
print("Roh-Datensatz:", raw.shape)

## 3. Vorverarbeitung mit **linearer Interpolation** (Paper-Imputation)

Wichtig: hier fuellen wir Fehlwerte per **zeitbasierter linearer Interpolation** je Station
(statt ffill/bfill wie in den anderen Notebooks) — so wie es dem Paper entspricht.

In [ ]:
df = raw.copy()
df["datetime"] = pd.to_datetime(df[["year","month","day","hour"]])
df = df.sort_values(["station","datetime"]).reset_index(drop=True)
df["time_idx"] = ((df["datetime"] - df["datetime"].min()).dt.total_seconds() // 3600).astype(int)
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek
df["wd"] = df["wd"].astype(str).replace("nan", "unknown").fillna("unknown")

# Punkte in Spaltennamen verboten -> PM2.5 zu PM25
df = df.rename(columns={"PM2.5": "PM25"})

# Alle numerischen Spalten je Station zeitbasiert linear interpolieren, Raender auffuellen
num_all = [c for c in ["PM25","PM10","SO2","NO2","CO","O3","TEMP","PRES","DEWP","RAIN","WSPM"] if c in df.columns]

def interp_station(g):
    g = g.set_index("datetime")
    g[num_all] = g[num_all].interpolate(method="time").ffill().bfill()
    return g.reset_index()

df = df.groupby("station", group_keys=False).apply(interp_station)
df = df.sort_values(["station", "time_idx"]).reset_index(drop=True)
print("Fehlende Werte nach Interpolation:", int(df[num_all].isna().sum().sum()))
df[["datetime","station","time_idx","PM25","PRES","WSPM","wd"]].head()

## 4. Paper-Protokoll: Zeitraeume und Variablen

In [ ]:
# Zeit-Aufteilung wie im Paper (Abschnitt 3.2.3)
TRAIN_START = pd.Timestamp("2013-03-01")
VAL_START   = pd.Timestamp("2015-03-01")
TEST_START  = pd.Timestamp("2016-03-01")   # Test = 2016-03-01 .. 2017-02-28

# Eingabevariablen wie im Paper (GA-Auswahl, Abschnitt 3.3): ohne NO2, CO, O3, TEMP
PAPER_REALS = ["PM25", "PM10", "SO2", "PRES", "DEWP", "RAIN", "WSPM"]
PAPER_CATS  = ["wd"]   # Windrichtung (kategorial)

# Paper-Stationen
PAPER_STATIONS = ["Dongsi", "Wanliu", "Changping"]

max_encoder_length    = 48   # Stunden Vergangenheit
max_prediction_length = 1    # 1-Stunde-Vorhersage (wie Paper)

print("Variablen:", PAPER_REALS, "+ kategorial", PAPER_CATS)
print("Stationen:", PAPER_STATIONS)

## 5. Trainings- und Test-Funktion je Station

Fuer jede Station wird ein **eigenes TFT** trainiert (nur die eigenen Kanaele dieser Station).
Trainiert wird auf Training+Validierung (bis kurz vor Testbeginn), bewertet auf dem **Test-Jahr**.

In [ ]:
def train_und_teste(station, max_epochs=25, verbose=True):
    d = df[df["station"] == station].sort_values("time_idx").reset_index(drop=True).copy()

    # Zeitindizes fuer die Schnitte
    test_start_idx = int(d.loc[d["datetime"] >= TEST_START, "time_idx"].min())
    val_len   = 30 * 24                       # letzte 30 Tage vor Test = interne Val (nur EarlyStopping)
    train_cut = test_start_idx - val_len      # davor wird trainiert (= Train+Val minus 30 Tage)

    ds_kwargs = dict(
        time_idx="time_idx", target="PM25", group_ids=["station"],
        max_encoder_length=max_encoder_length, max_prediction_length=max_prediction_length,
        time_varying_known_reals=["time_idx"],
        time_varying_unknown_reals=PAPER_REALS,
        time_varying_unknown_categoricals=PAPER_CATS,
        target_normalizer=GroupNormalizer(groups=["station"]),
        add_relative_time_idx=True, add_target_scales=True,
        add_encoder_length=True, allow_missing_timesteps=True,
    )

    training = TimeSeriesDataSet(d[d["time_idx"] < train_cut], **ds_kwargs)
    val_int  = TimeSeriesDataSet.from_dataset(training, d[d["time_idx"] < test_start_idx],
                                              min_prediction_idx=train_cut, stop_randomization=True)
    test     = TimeSeriesDataSet.from_dataset(training, d,
                                              min_prediction_idx=test_start_idx, stop_randomization=True)

    tl    = training.to_dataloader(train=True,  batch_size=128, num_workers=0)
    vl    = val_int.to_dataloader(train=False, batch_size=256, num_workers=0)
    testl = test.to_dataloader(train=False, batch_size=256, num_workers=0)

    model = TemporalFusionTransformer.from_dataset(
        training, learning_rate=0.03, hidden_size=32, attention_head_size=2,
        dropout=0.1, hidden_continuous_size=16, loss=QuantileLoss(),
        optimizer="adam", reduce_on_plateau_patience=3,
    )
    ckpt = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)
    trainer = pl.Trainer(
        max_epochs=max_epochs, accelerator="auto", devices=1, gradient_clip_val=0.1,
        callbacks=[EarlyStopping("val_loss", patience=4, mode="min"), ckpt],
        enable_progress_bar=verbose, logger=False,
    )
    trainer.fit(model, tl, vl)
    best = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path) if ckpt.best_model_path else model

    # Auf dem Test-Jahr auswerten
    p = best.predict(testl, mode="prediction", return_y=True)
    a = p.output.detach().cpu().numpy().reshape(-1)
    b = p.y[0].detach().cpu().numpy().reshape(-1)
    mae  = float(np.mean(np.abs(a - b)))
    rmse = float(np.sqrt(np.mean((a - b) ** 2)))
    print(f"[{station}]  Test-MAE = {mae:.2f} ug/m3 | RMSE = {rmse:.2f} | Test-Fenster: {len(b)}")
    return {"Station": station, "MAE": mae, "RMSE": rmse, "n_test": int(len(b))}

## 6. Alle drei Paper-Stationen trainieren

Hinweis: Es werden **drei** Modelle nacheinander trainiert — das dauert entsprechend laenger.

In [ ]:
ergebnisse = []
for st in PAPER_STATIONS:
    print(f"\n=== Trainiere Modell fuer Station: {st} ===")
    ergebnisse.append(train_und_teste(st))

res_df = pd.DataFrame(ergebnisse)
print("\nErgebnisse je Station (eigene Modelle, 1h-Vorhersage, Test-Jahr 2016/17):")
print(res_df.round(2).to_string(index=False))

## 7. Direkter Vergleich mit dem Paper

Die Paper-MAE-Werte stammen aus **Tsokov et al. (2022), Tables 3-5** — dem eigenen Modell des
Papers je Station, jeweils gemittelt ueber 10 Trainingslaeufe. Eingetragen ist der **beste der
drei Experimentierlaeufe** je Station (der Bereich der drei Laeufe steht als Kommentar). Alle
drei Werte sind bereits ausgefuellt. Der Vergleich ist so fair wie ohne Nachbau der
CNN-Architektur moeglich — die verbleibenden Unterschiede stehen in der Titelzelle.

In [1]:
# Paper-MAE (ug/m3), Quelle: Tsokov et al. 2022, Tables 3-5
# (bester der 3 Experimentierlaeufe, jeder ueber 10 Trainingslaeufe gemittelt)
paper_mae = {
    "Dongsi":    17.87,   # Table 3 (Bereich der 3 Laeufe: 17.9-19.1)
    "Wanliu":    15.37,   # Table 4 (Bereich: 15.4-16.8)
    "Changping": 15.16,   # Table 5 (Bereich: 15.2-16.1)
}

vgl = res_df[["Station", "MAE"]].rename(columns={"MAE": "MAE_unser_TFT"}).copy()
vgl["MAE_Paper"] = vgl["Station"].map(paper_mae)
vgl["Differenz"] = vgl["MAE_unser_TFT"] - vgl["MAE_Paper"]

print("1h-PM2.5-Vorhersage, MAE in ug/m3 (niedriger = besser):\n")
print(vgl.round(2).to_string(index=False))

# CSV-Export
out_path = os.path.join("..", "data", "ergebnis_paper_replikat_prostation.csv")
vgl.round(4).to_csv(out_path, index=False)
print("\nGespeichert unter:", out_path)

# Balkendiagramm (nur wo Paper-Wert eingetragen)
plot_vgl = vgl.dropna(subset=["MAE_Paper"])
if len(plot_vgl):
    x = np.arange(len(plot_vgl)); w = 0.4
    plt.figure(figsize=(7, 4))
    plt.bar(x - w/2, plot_vgl["MAE_unser_TFT"], w, label="Unser TFT (je Station)")
    plt.bar(x + w/2, plot_vgl["MAE_Paper"],     w, label="Paper (Tsokov 2022)")
    plt.xticks(x, plot_vgl["Station"]); plt.ylabel("MAE (ug/m3)")
    plt.title("1h-PM2.5: Per-Station-TFT vs. Paper")
    plt.legend(); plt.tight_layout(); plt.show()

NameError: name 'res_df' is not defined